# **Vision-based mmWave Beam Prediction in V2X Communications using JEPA**

# Loading libraries and pretraining task

In [30]:
from pathlib import Path
from google.colab import drive
import numpy as np
import pandas as pd
import glob
import matplotlib.pyplot as plt
import os
import sys
import random
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.models as models
import torchvision.transforms as transforms
import torch.nn.functional as F
from PIL import Image
import shutil
from collections import Counter, defaultdict
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

In [31]:
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### No need to run the next 4 cells if you have access to the directories. They have already been done

In [ ]:
#Move images to the Final_Dataset_Train folder -- This has already been done no need to run
input_root = '/content/drive/MyDrive/Colab Notebooks/ENEL 619 07/Final_Dataset_Train'
output_root = '/content/drive/MyDrive/Colab Notebooks/ENEL 619 07/IJEPA_Dataset'
os.makedirs(output_root, exist_ok=True)

total = 0

for scenario_folder in os.listdir(input_root):
    input_scenario = os.path.join(input_root, scenario_folder)
    if not os.path.isdir(input_scenario):
        continue
    output_scenario = os.path.join(output_root, scenario_folder)
    os.makedirs(output_scenario, exist_ok=True)
    images = [f for f in os.listdir(input_scenario) if f.endswith('.jpg')]
    for fname in images:
        shutil.copy2(os.path.join(input_scenario, fname), os.path.join(output_scenario, fname))
        total += 1
    print(f"Done {scenario_folder}: {len(images)} images")
print(f"\nTotal images copied: {total}")

#Move newly added scenario images to the Final_Dataset_Train folder
input_root = '/content/drive/MyDrive/Colab Notebooks/ENEL 619 07/Final_Dataset_Train/New Scenarios'
output_root = '/content/drive/MyDrive/Colab Notebooks/ENEL 619 07/IJEPA_Dataset'
os.makedirs(output_root, exist_ok=True)

total = 0

for scenario_folder in os.listdir(input_root):
    input_scenario = os.path.join(input_root, scenario_folder)
    if not os.path.isdir(input_scenario):
        continue
    output_scenario = os.path.join(output_root, scenario_folder)
    os.makedirs(output_scenario, exist_ok=True)
    images = [f for f in os.listdir(input_scenario) if f.endswith('.jpg')]
    for fname in images:
        shutil.copy2(os.path.join(input_scenario, fname), os.path.join(output_scenario, fname))
        total += 1
    print(f"Done {scenario_folder}: {len(images)} images")
print(f"\nTotal images copied: {total}")

In [ ]:
#Folder to store IJEPA checkpoints
os.makedirs('/content/drive/MyDrive/Colab Notebooks/ENEL 619 07/ijepa_checkpoints', exist_ok=True)
print("Done")

In [ ]:
#Change to this working directory
%cd '/content/drive/MyDrive/Colab Notebooks/ENEL 619 07/ijepa_modified_k'

In [ ]:
#Run main.py file to start pretraining
!python main.py --fname configs/deepsense6G_config.yaml --devices cuda:0

# Downstream Tasks - Beam prediction frozen encoder


In [32]:
## Add I-JEPA source code to Python path and importing the vit_base mode
sys.path.append("/content/drive/MyDrive/Colab Notebooks/ENEL 619 07/ijepa_modified_k")
from src.models.vision_transformer import vit_base

In [33]:
#Selecting scenarios and paths that contain images and the csv files
scenarios = [9]
base_dir = "/content/drive/MyDrive/Colab Notebooks/ENEL 619 07"
csv_root = os.path.join(base_dir, "Final Project dataset")
image_root = os.path.join(base_dir, "Final_Dataset_Train")
checkpoint_path = os.path.join(base_dir, "ijepa_checkpoints/deepsense6G_hpc-latest.pth.tar")
results_dir = os.path.join(base_dir, "results")
device = "cuda" if torch.cuda.is_available() else "cpu"
beam_cols = ["unit1_beam_index", "unit1_beam", "best_beam"]
top_k = [1, 2, 3, 5]
batch_size = 64
num_epochs = 100
lr = 1e-3
os.makedirs(results_dir, exist_ok=True) #Changes directory to that results_dir path
torch.manual_seed(42)# Set random seed for reproducibilit

In [34]:
def load_data():
    records = []
    for s in scenarios:
        csv_path = os.path.join(csv_root, f"scenario{s}", f"scenario{s}.csv")
        img_dir = os.path.join(image_root, f"camera_data_s{s}")
        if not os.path.exists(csv_path) or not os.path.exists(img_dir):
            print(f"[SKIP] Scenario {s}")
            continue
        df = pd.read_csv(csv_path)
        beam_col = next((c for c in beam_cols if c in df.columns), None)
        if beam_col is None or "unit1_rgb" not in df.columns:
            print(f"[SKIP] Scenario {s} - missing columns")
            continue
        df = df.dropna(subset=[beam_col, "unit1_rgb"])
        df["beam_label"] = df[beam_col].astype(int)
        df["img_path"] = df["unit1_rgb"].apply(lambda p: os.path.join(img_dir, os.path.basename(str(p))))
        df["scenario"] = s
        df = df[df["img_path"].apply(os.path.exists)]
        print(f"Scenario {s}: {len(df)} samples")
        records.append(df[["img_path", "beam_label", "scenario"]])
    return pd.concat(records, ignore_index=True)

In [35]:
def split(df):
    trains, tests = [], []
    for s in df["scenario"].unique():
        sub = df[df["scenario"] == s]
        n = int(len(sub) * 0.7)
        trains.append(sub.iloc[:n])
        tests.append(sub.iloc[n:])
    return pd.concat(trains), pd.concat(tests)

In [36]:
class BeamDataset(Dataset):
    def __init__(self, df, beam_to_idx, transform):
        self.df = df.reset_index(drop=True)
        self.beam_to_idx = beam_to_idx
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row["img_path"]).convert("RGB")
        return self.transform(img), self.beam_to_idx[row["beam_label"]]

#Loads frozen encoder from the latest checkpoint and freeze parameters
def load_encoder():
    encoder = vit_base(patch_size=16)
    ckpt = torch.load(checkpoint_path, map_location="cpu")
    sd = ckpt.get("target_encoder", ckpt.get("encoder", ckpt))
    sd = {k.replace("module.", ""): v for k, v in sd.items()}
    encoder.load_state_dict(sd, strict=False)
    for p in encoder.parameters():
        p.requires_grad = False
    encoder.eval()
    return encoder

In [37]:
@torch.no_grad()
def extract(encoder, loader):
    encoder.to(device)
    feats, labels = [], []
    for imgs, lbs in loader:
        out = encoder(imgs.to(device))
        feats.append((out.mean(dim=1) if out.dim() == 3 else out).cpu())
        labels.append(lbs)
    return torch.cat(feats), torch.cat(labels)

In [38]:
class Head(nn.Module):
    def __init__(self, in_dim, n):
        super().__init__()
        self.net = nn.Sequential(
            nn.LayerNorm(in_dim),
            nn.Linear(in_dim, 512),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(512, n))
    def forward(self, x):
        return self.net(x)

In [39]:
#Train MLP head on extracted features with class-weighted loss
def train_head(feats, labels, n):
    head = Head(feats.shape[1], n).to(device)
    w = labels.bincount(minlength=n).float().clamp(min=1)
    w = (w.sum() / (n * w)).to(device)
    opt = torch.optim.Adam(head.parameters(), lr=lr)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, num_epochs)
    criterion = nn.CrossEntropyLoss(weight=w)
    loader = DataLoader(torch.utils.data.TensorDataset(feats, labels), batch_size=256, shuffle=True)

    for epoch in range(1, num_epochs + 1):
        head.train()
        for xb, yb in loader:
            opt.zero_grad()
            criterion(head(xb.to(device)), yb.to(device)).backward()
            opt.step()
        sched.step()
        if epoch % 20 == 0 or epoch == num_epochs:
            print(f"Epoch {epoch}/{num_epochs}")
    head.eval()
    return head

In [40]:
def topk_acc(logits, labels):
    return {k: 100. * logits.topk(min(k, logits.shape[1]), dim=1).indices
                        .eq(labels.unsqueeze(1)).any(1).float().mean().item() for k in top_k}

def plot_topk(accs):
    fig, ax = plt.subplots(figsize=(7, 5))
    bars = ax.bar([f"Top-{k}" for k in top_k], [accs[k] for k in top_k],
                  color=["#2196F3", "#4CAF50", "#FF9800", "#F44336"], width=0.5)
    for bar, k in zip(bars, top_k):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.8,
                f"{accs[k]:.2f}%", ha="center", fontsize=11)
    ax.set_ylim(0, 110)
    ax.set_xlabel("Top-k")
    ax.set_ylabel("Accuracy (%)")
    ax.set_title(f"Beam Prediction Top-k - I-JEPA Frozen\nScenarios {scenarios}")
    ax.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(results_dir, "topk.png"), dpi=150)
    plt.close()

In [42]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),])

In [43]:
df = load_data()
train_df, test_df = split(df)
all_beams = sorted(df["beam_label"].unique())
beam_to_idx = {b: i for i, b in enumerate(all_beams)}
n_classes = len(all_beams)
train_loader = DataLoader(BeamDataset(train_df, beam_to_idx, transform),
                          batch_size=batch_size, shuffle=False, num_workers=16)
test_loader = DataLoader(BeamDataset(test_df, beam_to_idx, transform),
                         batch_size=batch_size, shuffle=False, num_workers=16)


Scenario 9: 5964 samples


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 16 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


In [44]:
# Load frozen encoder and extract features
encoder = load_encoder()
train_feats, train_labels = extract(encoder, train_loader)
test_feats, test_labels = extract(encoder, test_loader)

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 16 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


KeyboardInterrupt: 

In [ ]:
# Train MLP head on extracted features
head = train_head(train_feats, train_labels, n_classes)

with torch.no_grad():
    logits = head(test_feats.to(device)).cpu()

accs = topk_acc(logits, test_labels)
print("\n" + "="*40)
print(f"RESULTS - Scenarios {scenarios}")
print("="*40)
for k in top_k:
    print(f"Top-{k}: {accs[k]:.2f}%")
print("="*40)
plot_topk(accs)

In [ ]:
# Confusion matrix for top-20 most common beams
preds = logits.argmax(1).numpy()
labels_np = test_labels.numpy()
idx_to_beam = {v: k for k, v in beam_to_idx.items()}
counts = np.bincount(labels_np, minlength=len(beam_to_idx))
top_cls = np.intersect1d(np.argsort(counts)[::-1][:20], np.unique(labels_np))
mask = np.isin(labels_np, top_cls)
cm = confusion_matrix(labels_np[mask], preds[mask], labels=top_cls)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True).clip(min=1)

fig, ax = plt.subplots(figsize=(14, 11))
im = ax.imshow(cm_norm, cmap='Blues', vmin=0, vmax=1)
plt.colorbar(im, ax=ax, fraction=0.03, pad=0.04)
ax.set_xticks(range(len(top_cls)))
ax.set_yticks(range(len(top_cls)))
ax.set_xticklabels([str(idx_to_beam[c]) for c in top_cls], rotation=90, fontsize=8)
ax.set_yticklabels([str(idx_to_beam[c]) for c in top_cls], fontsize=8)
ax.set_xlabel('Predicted')
ax.set_ylabel('True')
ax.set_title('Confusion Matrix (Top-20 Beams, Normalized) - Frozen Encoder')
plt.tight_layout()
plt.savefig(os.path.join(results_dir, 'confusion_matrix_frozen.png'), dpi=150, bbox_inches='tight')
plt.show()

# Downstream Tasks - Beam prediction ResNet-18 baseline


In [45]:
base_dir = '/content/drive/MyDrive/Colab Notebooks/ENEL 619 07'
csv_root = os.path.join(base_dir, 'Final Project dataset')
image_root = os.path.join(base_dir, 'Final_Dataset_Train')
results_dir = os.path.join(base_dir, 'results')
os.makedirs(results_dir, exist_ok=True)
seed = 42
torch.manual_seed(seed)
np.random.seed(seed)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
batch_size = 128
epochs = 50
patience = 5
lr = 1e-3
weight_decay = 0.01
scenarios = [5]
beam_cols = ['best_beam', 'unit1_beam_index', 'unit1_beam']
img_cols = ['unit1_rgb', 'unit1_camera', 'image_path']

In [46]:
def load_data():
    all_dfs = []
    for s in scenarios:
        csv_files = glob.glob(os.path.join(csv_root, f'scenario{s}', '*.csv'))
        if not csv_files:
            continue
        df = pd.read_csv(csv_files[0])
        beam_col = next((c for c in beam_cols if c in df.columns), None)
        img_col = next((c for c in img_cols if c in df.columns), None)
        if beam_col is None or img_col is None:
            continue
        img_folder = os.path.join(image_root, f'camera_data_s{s}')
        if not os.path.exists(img_folder):
            continue
        available = set(os.listdir(img_folder))
        sdf = pd.DataFrame({
            'image_filename': df[img_col].apply(lambda x: os.path.basename(str(x))),
            'beam_index': df[beam_col].astype(int),
            'scenario': s,})
        sdf = sdf[sdf['image_filename'].isin(available)].reset_index(drop=True)
        all_dfs.append(sdf)
        print(f'Scenario {s}: {len(sdf)} samples')

    return pd.concat(all_dfs, ignore_index=True)

In [47]:
def prepare_labels(df):
    unique_beams = sorted(df['beam_index'].unique())
    beam_to_label = {b: i for i, b in enumerate(unique_beams)}
    df['label'] = df['beam_index'].map(beam_to_label)
    return df, len(unique_beams)

def split_data(df):
    n = len(df)
    indices = np.arange(n)
    rng = np.random.RandomState(seed)
    rng.shuffle(indices)
    train_end = int(n * 0.70)
    val_end = int(n * 0.85)
    train_df = df.iloc[indices[:train_end]].reset_index(drop=True)
    val_df = df.iloc[indices[train_end:val_end]].reset_index(drop=True)
    test_df = df.iloc[indices[val_end:]].reset_index(drop=True)

    return train_df, val_df, test_df

In [48]:
#Dataset for loading images and beam labels
class BeamDataset(Dataset):
    def __init__(self, dataframe, image_root, transform):
        self.df = dataframe.reset_index(drop=True)
        self.image_root = image_root
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        try:
            row = self.df.iloc[idx]
            img_path = os.path.join(self.image_root, f'camera_data_s{row["scenario"]}', row['image_filename'])
            img = Image.open(img_path).convert('RGB')
            img = self.transform(img)
            return img, row['label']
        except Exception:
            return self.__getitem__((idx + 1) % len(self.df))

In [49]:
def topk_accuracy(logits, labels, topk=(1, 2, 3, 5)):
    with torch.no_grad():
        maxk = max(topk)
        _, pred = logits.topk(maxk, 1, True, True)
        correct = pred.t().eq(labels.view(1, -1).expand_as(pred.t()))
        return {f'top{k}': correct[:k].reshape(-1).float().sum().item() / labels.size(0) * 100
                for k in topk}

def train_model(model, train_loader, val_loader, num_classes):
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)

    best_val_top3 = 0
    patience_ctr = 0

    for epoch in range(1, epochs + 1):
        model.train()
        total_loss, total_correct, n = 0, 0, 0
        for imgs, lbls in train_loader:
            imgs, lbls = imgs.to(device), lbls.to(device)
            logits = model(imgs)
            loss = criterion(logits, lbls)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * len(lbls)
            total_correct += (logits.argmax(1) == lbls).sum().item()
            n += len(lbls)

        model.eval()
        all_logits, all_labels = [], []
        with torch.no_grad():
            for imgs, lbls in val_loader:
                all_logits.append(model(imgs.to(device)).cpu())
                all_labels.append(lbls)
        val_logits = torch.cat(all_logits)
        val_labels = torch.cat(all_labels)
        val_res = topk_accuracy(val_logits, val_labels)

        if epoch % 10 == 0:
            print(f"Epoch {epoch}/{epochs} | Loss: {total_loss/n:.4f} | "
                  f"Val Top-3: {val_res['top3']:.2f}%")
        if val_res['top3'] > best_val_top3:
            best_val_top3 = val_res['top3']
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience_ctr = 0
        else:
            patience_ctr += 1
            if patience_ctr >= patience:
                print(f"Early stopping at epoch {epoch}")
                break

    return best_state

In [50]:
def plot_results(test_res, results_dir):
    fig, ax = plt.subplots(figsize=(6, 4))
    keys = ['top1', 'top2', 'top3', 'top5']
    values = [test_res[k] for k in keys]
    ax.bar(['Top-1', 'Top-2', 'Top-3', 'Top-5'], values, width=0.4, color='steelblue')
    ax.set_ylabel('Accuracy (%)')
    ax.set_title('Beam Prediction - ResNet-18')
    for i, v in enumerate(values):
        ax.text(i, v + 1, f'{v:.1f}%', ha='center')
    plt.tight_layout()
    plt.savefig(os.path.join(results_dir, 'resnet18_baseline.png'), dpi=150, bbox_inches='tight')
    plt.close()

In [51]:
train_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),])


eval_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),])

In [52]:
df = load_data()
df, num_classes = prepare_labels(df)
train_df, val_df, test_df = split_data(df)
print(f'Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}')
print(f'Classes: {num_classes}')

train_loader = DataLoader(BeamDataset(train_df, image_root, train_transform),
                          batch_size=batch_size, shuffle=True, num_workers=16, pin_memory=True)
val_loader = DataLoader(BeamDataset(val_df, image_root, eval_transform), batch_size=batch_size,
                        shuffle=False, num_workers=16, pin_memory=True)
test_loader = DataLoader(BeamDataset(test_df, image_root, eval_transform),
                         batch_size=batch_size, shuffle=False, num_workers=16, pin_memory=True)

Scenario 5: 2300 samples
Train: 1610, Val: 345, Test: 345
Classes: 61


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 16 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


In [53]:
model = models.resnet18(weights=None)
model.fc = nn.Linear(model.fc.in_features, num_classes)
model = model.to(device)
best_state = train_model(model, train_loader, val_loader, num_classes)
model.load_state_dict(best_state)
model = model.to(device)
model.eval()

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 16 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x79137915e200>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-

KeyboardInterrupt: 

In [ ]:
all_logits, all_labels = [], []
with torch.no_grad():
    for imgs, lbls in test_loader:
        all_logits.append(model(imgs.to(device)).cpu())
        all_labels.append(lbls)

test_logits = torch.cat(all_logits)
test_labels = torch.cat(all_labels)
test_res = topk_accuracy(test_logits, test_labels)

In [ ]:
print("TEST RESULTS - ResNet-18 Baseline")
print("=" * 40)
for k, v in test_res.items():
    print(f"{k.capitalize()}: {v:.2f}%")
print("=" * 40)
plot_results(test_res, results_dir)

# Downstream Tasks - NLOS/LOS I-JEPA

In [54]:
base_dir = '/content/drive/MyDrive/Colab Notebooks/ENEL 619 07'
image_root = os.path.join(base_dir, 'Final_Dataset_Train')
scenario_root = os.path.join(base_dir, 'Final Project dataset', 'New Scenarios')
checkpoints = os.path.join(base_dir, 'ijepa_checkpoints', 'deepsense6G_hpc-latest.pth.tar')
sys.path.insert(0, os.path.join(base_dir, 'ijepa_modified_k'))
seed = 42
torch.manual_seed(seed)
np.random.seed(seed)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
batch_size = 128

In [55]:
#Load Encoder from checkpoint
from src.models.vision_transformer import vit_base
encoder = vit_base().to(device)
checkpoint = torch.load(checkpoints, map_location='cpu')
state_dict = {k.replace('module.', ''): v for k, v in checkpoint['target_encoder'].items()}
encoder.load_state_dict(state_dict)
encoder.eval()
for p in encoder.parameters():
    p.requires_grad = False

In [ ]:
'''For scenario 24 and 30 — labels are binary (0 or 1) i.e. 0 for LOS and non-zero for NLOS
   The code below loads them and puts them into two classes
'''
all_rows = []

sc_30 = pd.read_csv(os.path.join(scenario_root, 'scenario30', 'scenario30.csv'))
label_dir_30 = os.path.join(scenario_root, 'scenario30', 'unit1', 'label_data')
img_col = 'unit1_rgb'
for _, row in sc_30.iterrows():
    fname = os.path.basename(str(row[img_col]))
    label_file = os.path.basename(str(row['unit1_label']))
    label_path = os.path.join(label_dir_30, label_file)
    if not os.path.exists(label_path):
        continue
    with open(label_path) as f:
        label = int(float(f.read().strip()))
    all_rows.append({'image_filename': fname, 'scenario': 30, 'label': min(label, 1)})

sc_24 = pd.read_csv(os.path.join(scenario_root, 'scenario24', 'scenario24.csv'))
label_dir_24 = os.path.join(scenario_root, 'scenario24', 'unit1', 'label_data')
for _, row in sc_24.iterrows():
    fname = os.path.basename(str(row[img_col]))
    label_file = os.path.basename(str(row['unit1_blockage']))
    label_path = os.path.join(label_dir_24, label_file)
    if not os.path.exists(label_path):
        continue
    with open(label_path) as f:
        val = int(float(f.read().strip()))
    label = 0 if val == 0 else 1
    all_rows.append({'image_filename': fname, 'scenario': 24, 'label': label})

full_df = pd.DataFrame(all_rows)
num_classes = 2

In [ ]:
rng = np.random.RandomState(seed)

# Split within each scenario by index
train_dfs, val_dfs, test_dfs = [], [], []
for s in [24, 30]:
    sdf = full_df[full_df['scenario'] == s].reset_index(drop=True)
    n = len(sdf)
    idx = np.arange(n)
    rng.shuffle(idx)
    train_end = int(n * 0.70)
    val_end = int(n * 0.85)
    train_dfs.append(sdf.iloc[idx[:train_end]])
    val_dfs.append(sdf.iloc[idx[train_end:val_end]])
    test_dfs.append(sdf.iloc[idx[val_end:]])
train_df = pd.concat(train_dfs, ignore_index=True)
val_df = pd.concat(val_dfs, ignore_index=True)
test_df = pd.concat(test_dfs, ignore_index=True)

In [ ]:
eval_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),])

class BlockageDataset(Dataset):
    def __init__(self, dataframe, image_root, transform):
        self.df = dataframe.reset_index(drop=True)
        self.image_root = image_root
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        try:
            row = self.df.iloc[idx]
            img_path = os.path.join(self.image_root, f'camera_data_s{row["scenario"]}', row['image_filename'])
            img = Image.open(img_path).convert('RGB')
            img = self.transform(img)
            return img, row['label']
        except Exception:
            return self.__getitem__((idx + 1) % len(self.df))


train_loader = DataLoader(BlockageDataset(train_df, image_root, eval_transform),
                          batch_size=batch_size, shuffle=False, pin_memory=True)
val_loader = DataLoader(BlockageDataset(val_df, image_root, eval_transform),
                        batch_size=batch_size, shuffle=False, pin_memory=True)
test_loader = DataLoader(BlockageDataset(test_df, image_root, eval_transform),
                         batch_size=batch_size, shuffle=False, pin_memory=True)

In [ ]:
@torch.no_grad()
def extract_features(encoder, dataloader, device):
    features, labels = [], []
    for imgs, lbls in dataloader:
        out = encoder(imgs.to(device))
        if isinstance(out, (list, tuple)):
            out = out[0]
        features.append(out.mean(dim=1).cpu())
        labels.append(lbls)
    return torch.cat(features), torch.cat(labels)

train_features, train_labels = extract_features(encoder, train_loader, device)
val_features, val_labels = extract_features(encoder, val_loader, device)
test_features, test_labels = extract_features(encoder, test_loader, device)
embed_dim = train_features.shape[1]

In [ ]:
#Linear classification head on frozen I-JEPA features
head = nn.Sequential(
    nn.LayerNorm(embed_dim),
    nn.Linear(embed_dim, num_classes),).to(device)

n_los = (train_labels == 0).sum().item()
n_nlos = (train_labels == 1).sum().item()
weights = torch.tensor([len(train_labels) / (2 * n_los), len(train_labels) / (2 * n_nlos)]).to(device)
criterion = nn.CrossEntropyLoss(weight=weights)
optimizer = torch.optim.AdamW(head.parameters(), lr=1e-3, weight_decay=1e-4)

epochs = 50
patience = 5
best_val_acc = 0
patience_ctr = 0

for epoch in range(1, epochs + 1):
    head.train()
    idx = torch.randperm(len(train_features))
    total_loss = 0
    for start in range(0, len(train_features), batch_size):
        batch_idx = idx[start:start + batch_size]
        logits = head(train_features[batch_idx].to(device))
        loss = criterion(logits, train_labels[batch_idx].to(device))
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(batch_idx)
    head.eval()
    with torch.no_grad():
        val_logits = head(val_features.to(device))
    val_preds = val_logits.argmax(1).cpu()
    val_acc = (val_preds == val_labels).float().mean().item() * 100

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_state = {k: v.clone() for k, v in head.state_dict().items()}
        patience_ctr = 0
    else:
        patience_ctr += 1
        if patience_ctr >= patience:
            break

In [ ]:
head.load_state_dict(best_state)
head.eval()
with torch.no_grad():
    test_logits = head(test_features.to(device))
test_preds = test_logits.argmax(1).cpu()
test_acc = (test_preds == test_labels).float().mean().item() * 100
print(f'Test Accuracy: {test_acc:.2f}%\n')
#Classification metrics for LOS vs NLOS
print(classification_report(test_labels.numpy(), test_preds.numpy(),target_names=['LOS', 'NLOS'], digits=3))

In [ ]:
# Confusion matrix to visualize prediction performance across classes
cm = confusion_matrix(test_labels.numpy(), test_preds.numpy(), normalize='true')
fig, ax = plt.subplots(figsize=(4, 4))
cmd = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['LOS', 'NLOS'])
cmd.plot(ax=ax, cmap='Blues', values_format='.2f')
ax.set_xlabel('Predicted')
ax.set_ylabel('True')
ax.set_title('LOS/NLOS - I-JEPA')
cbar = cmd.im_.colorbar
cbar.ax.set_aspect('auto')
plt.tight_layout()
plt.show()

# Downstream Tasks - NLOS/LOS ResNet-18 baseline


In [ ]:
#Point to directories containing relevant data and change directory
base_dir = '/content/drive/MyDrive/Colab Notebooks/ENEL 619 07'
image_root = os.path.join(base_dir, 'Final_Dataset_Train')
scenario_root = os.path.join(base_dir, 'Final Project dataset', 'New Scenarios')
results_dir = os.path.join(base_dir, 'results')
os.makedirs(results_dir, exist_ok=True)
seed = 42
torch.manual_seed(seed)
np.random.seed(seed)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
batch_size = 128
epochs = 50
lr = 1e-3
weight_decay = 0.01
scenarios = [24, 30] #Scenarios 24 and 30 used for NLOS/LOS

In [ ]:
def load_data():
    all_rows = []
    img_col = 'unit1_rgb'

    scenarios = [(30, 'unit1_label'),(24, 'unit1_blockage'),]

    for scenario_id, label_col in scenarios:
        df = pd.read_csv(
            os.path.join(scenario_root, f'scenario{scenario_id}', f'scenario{scenario_id}.csv') )
        label_dir = os.path.join(
            scenario_root, f'scenario{scenario_id}', 'unit1', 'label_data' )

        for _, row in df.iterrows():
            fname = os.path.basename(str(row[img_col]))
            label_file = os.path.basename(str(row[label_col]))
            label_path = os.path.join(label_dir, label_file)

            if not os.path.exists(label_path):
                continue
            try:
                with open(label_path) as f:
                    val = int(float(f.read().strip()))
                all_rows.append({
                    'image_filename': fname,
                    'scenario': scenario_id,
                    'label': 0 if val == 0 else 1})
            except:
                continue
    full_df = pd.DataFrame(all_rows)
    print(f'Total: {len(full_df)}')
    print(f'LOS (0): {(full_df["label"] == 0).sum()}')
    print(f'NLOS (1): {(full_df["label"] == 1).sum()}')

    return full_df

In [ ]:
def split_data(df):
    train_dfs, test_dfs = [], []
    for s in df["scenario"].unique():
        sub = df[df["scenario"] == s]
        n_train = int(len(sub) * 0.7)
        train_dfs.append(sub.iloc[:n_train])
        test_dfs.append(sub.iloc[n_train:])
    return pd.concat(train_dfs, ignore_index=True), pd.concat(test_dfs, ignore_index=True)

In [ ]:
class LOSNLOSDataset(Dataset):
    def __init__(self, df, image_root, transform):
        self.df = df.reset_index(drop=True)
        self.image_root = image_root
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = os.path.join(self.image_root, f'camera_data_s{row["scenario"]}', row['image_filename'])
        try:
            img = Image.open(img_path).convert("RGB")
        except:
            img = Image.new("RGB", (224, 224))
        return self.transform(img), row["label"]

In [ ]:
def train_epoch(model, loader, criterion, optimizer):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
        _, preds = outputs.max(1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
    return running_loss / len(loader), 100.0 * correct / total

In [ ]:
def evaluate(model, loader):"
    model.eval()
    all_preds, all_labels = [], []

    with torch.no_grad():
        for imgs, labels in loader:
            outputs = model(imgs.to(device))
            _, preds = outputs.max(1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.numpy())
    accuracy = 100.0 * np.mean(np.array(all_preds) == np.array(all_labels))
    return accuracy, np.array(all_preds), np.array(all_labels)

In [ ]:
def train_model(model, train_loader, test_loader):
    all_labels = torch.tensor([y for _, y in train_loader.dataset])
    counts = torch.bincount(all_labels, minlength=2).float()
    weights = (counts.sum() / (2 * counts.clamp(min=1))).to(device)
    criterion = nn.CrossEntropyLoss(weight=weights)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, epochs)

    best_acc = 0.0

    for epoch in range(1, epochs + 1):
        train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer)
        test_acc, _, _ = evaluate(model, test_loader)
        scheduler.step()
        if test_acc > best_acc:
            best_acc = test_acc
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        if epoch % 10 == 0:
            print(f"Epoch {epoch}/{epochs} | Loss: {train_loss:.4f} | "
                  f"Train: {train_acc:.2f}% | Test: {test_acc:.2f}%")
    model.load_state_dict(best_state)
    final_acc, preds, labels = evaluate(model, test_loader)
    return model, final_acc, preds, labels

In [ ]:
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(0.2, 0.2, 0.2, 0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),])

test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),])

In [ ]:
df = load_data()
train_df, test_df = split_data(df)
print(f'\nTrain: {len(train_df)} | Test: {len(test_df)}')


train_loader = DataLoader(LOSNLOSDataset(train_df, image_root, train_transform),
                          batch_size=batch_size, shuffle=True, num_workers=16)
test_loader = DataLoader(LOSNLOSDataset(test_df, image_root, test_transform),
                         batch_size=batch_size, shuffle=False, num_workers=16)

In [ ]:
model = models.resnet18(weights=None)
model.fc = nn.Linear(model.fc.in_features, 2)
model = model.to(device)

NameError: name 'models' is not defined

In [ ]:
model, final_acc, preds, labels = train_model(model, train_loader, test_loader)
print("\n")
print("LOS/NLOS - ResNet-18 Baseline")
print(f"Accuracy: {final_acc:.2f}%")
print("="*40)
print("\nClassification Report:")
print(classification_report(labels, preds, target_names=['LOS', 'NLOS']))

cm = confusion_matrix(labels, preds, normalize='true')
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['LOS', 'NLOS'])
fig, ax = plt.subplots(figsize=(5, 5))
disp.plot(ax=ax, cmap='Blues', colorbar=True)
ax.set_title('LOS/NLOS - ResNet-18')
plt.tight_layout()
plt.savefig(os.path.join(results_dir, 'losnlos_resnet18_cm.png'), dpi=150, bbox_inches='tight')
plt.show()

#  Downstream Tasks - Day/Night I-JEPA


In [ ]:
base_dir = '/content/drive/MyDrive/Colab Notebooks/ENEL 619 07'
image_root = os.path.join(base_dir, 'Final_Dataset_Train')
checkpoint_path = os.path.join(base_dir, 'ijepa_checkpoints/deepsense6G_hpc-latest.pth.tar')
results_dir = os.path.join(base_dir, 'results')
os.makedirs(results_dir, exist_ok=True)
seed = 42
torch.manual_seed(seed)
np.random.seed(seed)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
batch_size = 128
epochs = 50
patience = 5
lr = 1e-3
weight_decay = 1e-4

sys.path.insert(0, os.path.join(base_dir, 'ijepa_modified_k'))
from src.models.vision_transformer import vit_base

In [ ]:
# Day/Night scenario labels
night_scenarios = {2, 4, 5, 14, 33, 34}
day_scenarios = {1, 3, 6, 7, 8, 9}

In [ ]:
def load_data():
    all_scenarios = sorted(night_scenarios | day_scenarios)
    all_rows = []

    for s in all_scenarios:
        img_folder = os.path.join(image_root, f'camera_data_s{s}')
        if not os.path.exists(img_folder):
            continue
        for fname in os.listdir(img_folder):
            if fname.endswith('.jpg'):
                all_rows.append({
                    'image_filename': fname,
                    'scenario': s,
                    'label': 1 if s in night_scenarios else 0,})

    full_df = pd.DataFrame(all_rows)
    print(f'Total: {len(full_df)}, Day: {(full_df["label"]==0).sum()}, Night: {(full_df["label"]==1).sum()}')
    return full_df

In [ ]:
def split_by_scenario(df):
    rng = np.random.RandomState(seed)
    day_list = sorted(day_scenarios)
    night_list = sorted(night_scenarios)
    rng.shuffle(day_list)
    rng.shuffle(night_list)

    train_scenarios = set(day_list[:4] + night_list[:4])
    val_scenarios = set(day_list[4:5] + night_list[4:5])
    test_scenarios = set(day_list[5:] + night_list[5:])

    train_df = df[df['scenario'].isin(train_scenarios)].reset_index(drop=True)
    val_df = df[df['scenario'].isin(val_scenarios)].reset_index(drop=True)
    test_df = df[df['scenario'].isin(test_scenarios)].reset_index(drop=True)
    print(f'Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}')
    return train_df, val_df, test_df

In [ ]:
#Load pretrained encoder and freeze all weights
def load_encoder():
    encoder = vit_base().to(device)
    checkpoint = torch.load(checkpoint_path, map_location='cpu')
    state_dict = {k.replace('module.', ''): v for k, v in checkpoint['target_encoder'].items()}
    encoder.load_state_dict(state_dict)
    encoder.eval()
    for p in encoder.parameters():
        p.requires_grad = False
    return encoder

@torch.no_grad()
def extract_features(encoder, dataloader):
    features, labels = [], []
    for imgs, lbls in dataloader:
        out = encoder(imgs.to(device))
        if isinstance(out, (list, tuple)):
            out = out[0]
        features.append(out.mean(dim=1).cpu())
        labels.append(lbls)
    return torch.cat(features), torch.cat(labels)

In [ ]:
def train_head(train_features, train_labels, val_features, val_labels, embed_dim):
    head = nn.Sequential(
        nn.LayerNorm(embed_dim),
        nn.Linear(embed_dim, 2),).to(device)

    n_day = (train_labels == 0).sum().item()
    n_night = (train_labels == 1).sum().item()
    weights = torch.tensor([len(train_labels) / (2 * n_day), len(train_labels) / (2 * n_night)]).to(device)

    criterion = nn.CrossEntropyLoss(weight=weights)
    optimizer = torch.optim.AdamW(head.parameters(), lr=lr, weight_decay=weight_decay)

    best_val_acc = 0
    patience_ctr = 0

    for epoch in range(1, epochs + 1):
        head.train()
        idx = torch.randperm(len(train_features))
        total_loss = 0

        for start in range(0, len(train_features), batch_size):
            batch_idx = idx[start:start + batch_size]
            logits = head(train_features[batch_idx].to(device))
            loss = criterion(logits, train_labels[batch_idx].to(device))
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * len(batch_idx)

        head.eval()
        with torch.no_grad():
            val_logits = head(val_features.to(device))
        val_acc = (val_logits.argmax(1) == val_labels.to(device)).float().mean().item() * 100

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state = {k: v.clone() for k, v in head.state_dict().items()}
            patience_ctr = 0
        else:
            patience_ctr += 1
            if patience_ctr >= patience:
                print(f"Early stopping at epoch {epoch}")
                break

        if epoch % 10 == 0:
            print(f"Epoch {epoch}/{epochs} | Val Acc: {val_acc:.2f}%")

    head.load_state_dict(best_state)
    return head


In [ ]:
def evaluate(head, test_features, test_labels):
    head.eval()
    with torch.no_grad():
        test_logits = head(test_features.to(device))
    test_preds = test_logits.argmax(1).cpu()
    test_acc = (test_preds == test_labels).float().mean().item() * 100
    return test_acc, test_preds

    # Image preprocessing: resize to 256x256, center crop to 224x224, normalize with ImageNet stats
    transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),])

In [ ]:
df = load_data()
train_df, val_df, test_df = split_by_scenario(df)
train_loader = DataLoader(DayNightDataset(train_df, image_root, transform),
                          batch_size=batch_size, shuffle=False, num_workers=16, pin_memory=True)
val_loader = DataLoader(DayNightDataset(val_df, image_root, transform),
                        batch_size=batch_size, shuffle=False, num_workers=16, pin_memory=True)
test_loader = DataLoader(DayNightDataset(test_df, image_root, transform),
                         batch_size=batch_size, shuffle=False, num_workers=16, pin_memory=True)

In [ ]:
encoder = load_encoder()
train_features, train_labels = extract_features(encoder, train_loader)
val_features, val_labels = extract_features(encoder, val_loader)
test_features, test_labels = extract_features(encoder, test_loader)
embed_dim = train_features.shape[1]

In [ ]:
head = train_head(train_features, train_labels, val_features, val_labels, embed_dim)
test_acc, test_preds = evaluate(head, test_features, test_labels)

print("\n" + "="*40)
print("TEST RESULTS - Day/Night I-JEPA")
print("="*40)
print(f"Accuracy: {test_acc:.2f}%")
print("="*40)
print("\nClassification Report:")
print(classification_report(test_labels.numpy(), test_preds.numpy(), target_names=['Day', 'Night'], digits=3))
print("\nDone. Results saved to:", results_dir)

cm = confusion_matrix(test_labels.numpy(), test_preds.numpy(), normalize='true')
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Day', 'Night'])
fig, ax = plt.subplots(figsize=(5, 5))
disp.plot(ax=ax, cmap='Greens', colorbar=True)
ax.set_title('Day/Night - I-JEPA')
plt.tight_layout()
plt.savefig(os.path.join(results_dir, 'daynight_ijepa_cm.png'), dpi=150, bbox_inches='tight')
plt.show()


# Downstream Tasks - Day/Night ResNet-18 baseline

In [ ]:
#Point to directories containing relevant data and change directory
base_dir = '/content/drive/MyDrive/Colab Notebooks/ENEL 619 07'
image_root = os.path.join(base_dir, 'Final_Dataset_Train')
results_dir = os.path.join(base_dir, 'results')
os.makedirs(results_dir, exist_ok=True)
seed = 42
torch.manual_seed(seed)
np.random.seed(seed)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
batch_size = 128
epochs = 50
lr = 1e-4
weight_decay = 1e-4

In [ ]:
# Day/Night scenario labels
night_scenarios = {2, 4, 5, 14, 33, 34}
day_scenarios = {1, 3, 6, 7, 8, 9}

In [ ]:
#Load day/night labeled images from all scenarios
def load_data():
    all_scenarios = sorted(night_scenarios | day_scenarios)
    records = []

    for s in all_scenarios:
        img_folder = os.path.join(image_root, f'camera_data_s{s}')
        if not os.path.exists(img_folder):
            continue
        for fname in os.listdir(img_folder):
            if fname.lower().endswith(('.jpg')):
                records.append({
                    'img_path': os.path.join(img_folder, fname),
                    'label': 1 if s in night_scenarios else 0,
                    'scenario': s })

    full_df = pd.DataFrame(records)
    print(f'Total: {len(full_df)}, Day: {(full_df["label"]==0).sum()}, Night: {(full_df["label"]==1).sum()}')
    return full_df

In [ ]:
def split_by_scenario(df):
    train_dfs, test_dfs = [], []
    for s in df["scenario"].unique():
        sub = df[df["scenario"] == s]
        n_train = int(len(sub) * 0.7)
        train_dfs.append(sub.iloc[:n_train])
        test_dfs.append(sub.iloc[n_train:])
    return pd.concat(train_dfs, ignore_index=True), pd.concat(test_dfs, ignore_index=True)

#Dataset for day/night classification
class DayNightDataset(Dataset):
    def __init__(self, df, transform):
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        try:
            img = Image.open(row["img_path"]).convert("RGB")
        except:
            img = Image.new("RGB", (224, 224))
        return self.transform(img), row["label"]

In [ ]:
def train_epoch(model, loader, criterion, optimizer):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
        _, preds = outputs.max(1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
    return running_loss / len(loader), 100.0 * correct / total


def evaluate(model, loader):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for imgs, labels in loader:
            outputs = model(imgs.to(device))
            _, preds = outputs.max(1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.numpy())
    accuracy = 100.0 * np.mean(np.array(all_preds) == np.array(all_labels))
    return accuracy, np.array(all_preds), np.array(all_labels)

In [ ]:
def train_model(model, train_loader, test_loader):
    all_labels = torch.cat([y for _, y in train_loader])
    counts = torch.bincount(all_labels, minlength=2).float()
    weights = (counts.sum() / (2 * counts.clamp(min=1))).to(device)
    criterion = nn.CrossEntropyLoss(weight=weights)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, epochs)

    best_acc = 0.0

    for epoch in range(1, epochs + 1):
        train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer)
        test_acc, _, _ = evaluate(model, test_loader)
        scheduler.step()

        if test_acc > best_acc:
            best_acc = test_acc
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

        if epoch % 10 == 0:
            print(f"Epoch {epoch}/{epochs} | Loss: {train_loss:.2f} | "
                  f"Train: {train_acc:.2f}% | Test: {test_acc:.2f}%")

    model.load_state_dict(best_state)
    final_acc, preds, labels = evaluate(model, test_loader)
    return model, final_acc, preds, labels

In [ ]:
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(0.2, 0.2, 0.2, 0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),])

test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),])

In [ ]:
df = load_data()

#Splot by scenario
train_df, test_df = split_by_scenario(df)
train_loader = DataLoader(DayNightDataset(train_df, train_transform), batch_size=batch_size, shuffle=True, num_workers=16)
test_loader = DataLoader(DayNightDataset(test_df, test_transform), batch_size=batch_size, shuffle=False, num_workers=16)

In [ ]:
#Initialize ResNet-18
model = models.resnet18(pretrained=False)
model.fc = nn.Linear(model.fc.in_features, 2)
model = model.to(device)

In [ ]:
model, final_acc, preds, labels = train_model(model, train_loader, test_loader)
print("\n" + "="*40)
print("Results- ResNet-18 Baseline")
print("="*40)
print(f"Accuracy: {final_acc:.2f}%")
print("="*40)
print("\nClassification Day/Night:")
print(classification_report(labels, preds, target_names=['Day', 'Night']))
print("\nDone. Results saved to:", results_dir)

cm = confusion_matrix(labels, preds, normalize='true')
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Day', 'Night'])
fig, ax = plt.subplots(figsize=(5, 5))
disp.plot(ax=ax, cmap='Blues', colorbar=True)
ax.set_title('Day/Night - ResNet-18')
plt.tight_layout()
plt.savefig(os.path.join(results_dir, 'daynight_resnet18_cm.png'), dpi=150, bbox_inches='tight')
plt.show()

# Downstream Tasks - Beam Prediction LoRA Fine-tuning

In [ ]:
base_dir = '/content/drive/MyDrive/Colab Notebooks/ENEL 619 07'
csv_root = os.path.join(base_dir, 'Final Project dataset')
image_root = os.path.join(base_dir, 'Final_Dataset_Train')
checkpoint_path = os.path.join(base_dir, 'ijepa_checkpoints/deepsense6G_hpc-latest.pth.tar')
results_dir = os.path.join(base_dir, 'results')
os.makedirs(results_dir, exist_ok=True)
seed = 42
torch.manual_seed(seed)
np.random.seed(seed)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
batch_size = 128
epochs = 50
patience = 5
lr_lora = 1e-4
lr_head = 1e-3
weight_decay = 0.05

sys.path.insert(0, os.path.join(base_dir, 'ijepa_modified_k'))
from src.models.vision_transformer import vit_base

In [ ]:
scenarios = [5] #Scenario can be changed

#Column names for beam labels in CSV files and column names for image paths in CSV files
beam_cols = ['best_beam', 'unit1_beam_index', 'unit1_beam']
img_cols = ['unit1_rgb', 'unit1_camera', 'image_path']

In [ ]:
# LoRA hyperparameters 
lora_rank = 32
lora_alpha = 32
lora_num_layers = 12

class LoRALinear(nn.Module):
    def __init__(self, original, rank, alpha):
        super().__init__()
        self.original = original
        self.lora_A = nn.Parameter(torch.randn(rank, original.in_features) * 0.01)
        self.lora_B = nn.Parameter(torch.zeros(original.out_features, rank))
        self.scale = alpha / rank

    def forward(self, x):
        return self.original(x) + self.scale * (x @ self.lora_A.T) @ self.lora_B.T

def inject_lora(encoder, rank, alpha, num_layers):
    lora_params = []
    n_blocks = len(encoder.blocks)

    for i in range(n_blocks - num_layers, n_blocks):
        block = encoder.blocks[i]

        lora_qkv = LoRALinear(block.attn.qkv, rank, alpha)
        block.attn.qkv = lora_qkv
        lora_params.extend([lora_qkv.lora_A, lora_qkv.lora_B])

        lora_proj = LoRALinear(block.attn.proj, rank, alpha)
        block.attn.proj = lora_proj
        lora_params.extend([lora_proj.lora_A, lora_proj.lora_B])

    return lora_params

In [ ]:
def load_data():
    all_dfs = []
    for s in scenarios:
        csv_files = glob.glob(os.path.join(csv_root, f'scenario{s}', '*.csv'))
        if not csv_files:
            continue
        df = pd.read_csv(csv_files[0])
        beam_col = next((c for c in beam_cols if c in df.columns), None)
        img_col = next((c for c in img_cols if c in df.columns), None)
        if beam_col is None or img_col is None:
            continue
        img_folder = os.path.join(image_root, f'camera_data_s{s}')
        if not os.path.exists(img_folder):
            continue
        available = set(os.listdir(img_folder))

        sdf = pd.DataFrame({
            'image_filename': df[img_col].apply(lambda x: os.path.basename(str(x))),
            'beam_index': df[beam_col].astype(int), 'scenario': s,})
        sdf = sdf[sdf['image_filename'].isin(available)].reset_index(drop=True)
        all_dfs.append(sdf)
        print(f'Scenario {s}: {len(sdf)} samples')
    return pd.concat(all_dfs, ignore_index=True)

def prepare_labels(df):
    unique_beams = sorted(df['beam_index'].unique())
    beam_to_label = {b: i for i, b in enumerate(unique_beams)}
    df['label'] = df['beam_index'].map(beam_to_label)
    return df, len(unique_beams)

In [ ]:
def split_data(df):
    n = len(df)
    indices = np.arange(n)
    rng = np.random.RandomState(seed)
    rng.shuffle(indices)
    train_end = int(n * 0.70)
    val_end = int(n * 0.85)
    train_df = df.iloc[indices[:train_end]].reset_index(drop=True)
    val_df = df.iloc[indices[train_end:val_end]].reset_index(drop=True)
    test_df = df.iloc[indices[val_end:]].reset_index(drop=True)
    return train_df, val_df, test_df

class BeamDataset(Dataset):
    def __init__(self, dataframe, image_root, transform):
        self.df = dataframe.reset_index(drop=True)
        self.image_root = image_root
        self.transform = transform
    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        try:
            row = self.df.iloc[idx]
            img_path = os.path.join(self.image_root, f'camera_data_s{row["scenario"]}', row['image_filename'])
            img = Image.open(img_path).convert('RGB')
            return self.transform(img), row['label']
        except Exception:
            return self.__getitem__((idx + 1) % len(self.df))

In [ ]:
#Beam prediction model with LoRA-adapted encoder
class BeamPredictor(nn.Module):
    def __init__(self, encoder, num_classes, embed_dim=768):
        super().__init__()
        self.encoder = encoder
        self.head = nn.Sequential(
            nn.LayerNorm(embed_dim),
            nn.Linear(embed_dim, 512),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(512, num_classes),)

    def forward(self, x):
        out = self.encoder(x)
        if isinstance(out, (list, tuple)):
            out = out[0]
        out = out.mean(dim=1)
        return self.head(out)

def load_encoder():
    encoder = vit_base().to(device)
    checkpoint = torch.load(checkpoint_path, map_location='cpu')
    state_dict = {k.replace('module.', ''): v for k, v in checkpoint['target_encoder'].items()}
    encoder.load_state_dict(state_dict)

    for p in encoder.parameters():
        p.requires_grad = False
    return encoder

In [ ]:
def topk_accuracy(logits, labels, topk=(1, 2, 3, 5)):
    with torch.no_grad():
        maxk = max(topk)
        _, pred = logits.topk(maxk, 1, True, True)
        correct = pred.t().eq(labels.view(1, -1).expand_as(pred.t()))
        return {f'top{k}': correct[:k].reshape(-1).float().sum().item() / labels.size(0) * 100
                for k in topk}

In [ ]:
def train_model(model, lora_params, train_loader, val_loader):
    optimizer = torch.optim.AdamW([
        {'params': lora_params, 'lr': lr_lora},
        {'params': model.head.parameters(), 'lr': lr_head},], weight_decay=weight_decay)

    criterion = nn.CrossEntropyLoss()
    best_val_top3 = 0
    patience_ctr = 0

    for epoch in range(1, epochs + 1):
        model.train()
        total_loss, total_correct, n = 0, 0, 0

        for imgs, lbls in train_loader:
            imgs, lbls = imgs.to(device), lbls.to(device)
            logits = model(imgs)
            loss = criterion(logits, lbls)
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(lora_params + list(model.head.parameters()), 1.0)
            optimizer.step()

            total_loss += loss.item() * len(lbls)
            total_correct += (logits.argmax(1) == lbls).sum().item()
            n += len(lbls)

        model.eval()
        all_logits, all_labels = [], []
        with torch.no_grad():
            for imgs, lbls in val_loader:
                all_logits.append(model(imgs.to(device)).cpu())
                all_labels.append(lbls)

        val_logits = torch.cat(all_logits)
        val_labels = torch.cat(all_labels)
        val_res = topk_accuracy(val_logits, val_labels)

        if epoch % 10 == 0:
            print(f"Epoch {epoch}/{epochs} | Loss: {total_loss/n:.4f} | "
                  f"Train: {total_correct/n*100:.1f}% | Val Top-3: {val_res['top3']:.1f}%")

        if val_res['top3'] > best_val_top3:
            best_val_top3 = val_res['top3']
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience_ctr = 0
        else:
            patience_ctr += 1
            if patience_ctr >= patience:
                print(f"Early stopping at epoch {epoch}")
                break
    return best_state

In [ ]:
def plot_results(test_res):
    fig, ax = plt.subplots(figsize=(6, 4))
    values = [test_res[k] for k in ['top1', 'top2', 'top3', 'top5']]
    ax.bar(['Top-1', 'Top-2', 'Top-3', 'Top-5'], values, width=0.4, color='steelblue')
    ax.set_ylabel('Accuracy (%)')
    ax.set_title('Beam Prediction - LoRA Fine-Tuning')
    for i, v in enumerate(values):
        ax.text(i, v + 1, f'{v:.1f}%', ha='center')
    plt.tight_layout()
    plt.savefig(os.path.join(results_dir, 'beam_lora.png'), dpi=150, bbox_inches='tight')
    plt.close()

In [ ]:
train_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),])

eval_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),])

In [ ]:
df = load_data()

df, num_classes = prepare_labels(df)
train_df, val_df, test_df = split_data(df)
print(f'Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}, Classes: {num_classes}')

train_loader = DataLoader(BeamDataset(train_df, image_root, train_transform),
                          batch_size=batch_size, shuffle=True, num_workers=16, pin_memory=True)
val_loader = DataLoader(BeamDataset(val_df, image_root, eval_transform),
                        batch_size=batch_size, shuffle=False, num_workers=16, pin_memory=True)
test_loader = DataLoader(BeamDataset(test_df, image_root, eval_transform),
                         batch_size=batch_size, shuffle=False, num_workers=16, pin_memory=True)

In [ ]:
#Load encoder, add the LoRA
encoder = load_encoder()
lora_params = inject_lora(encoder, rank, alpha, num_layers)
model = BeamPredictor(encoder, num_classes).to(device)

best_state = train_model(model, lora_params, train_loader, val_loader)

In [ ]:
model.load_state_dict(best_state)
model = model.to(device)
model.eval()

all_logits, all_labels = [], []
with torch.no_grad():
    for imgs, lbls in test_loader:
        all_logits.append(model(imgs.to(device)).cpu())
        all_labels.append(lbls)
test_logits = torch.cat(all_logits)
test_labels = torch.cat(all_labels)
test_res = topk_accuracy(test_logits, test_labels)

In [ ]:
print("\n" + "=" * 40)
print("LoRA Fine-Tuning - Test Results")
print("=" * 40)
for k in ['top1', 'top2', 'top3', 'top5']:
    print(f"Top-{k[-1]}: {test_res[k]:.2f}%")
print("=" * 40)

plot_results(test_res)

# Downstream Tasks - Beam Prediction Unfreezing the last N-Layers
N=4

In [ ]:
base_dir = '/content/drive/MyDrive/Colab Notebooks/ENEL 619 07'
csv_root = os.path.join(base_dir, 'Final Project dataset')
image_root = os.path.join(base_dir, 'Final_Dataset_Train')
checkpoint_path = os.path.join(base_dir, 'ijepa_checkpoints/deepsense6G_hpc-latest.pth.tar')
results_dir = os.path.join(base_dir, 'results')
os.makedirs(results_dir, exist_ok=True)
seed = 42
torch.manual_seed(seed)
np.random.seed(seed)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
batch_size = 128
epochs = 100
lr_backbone = 1e-6
lr_head = 1e-3
weight_decay = 0.05
num_unfreeze = 4    # Number of last transformer blocks to unfreeze
top_k = [1, 2, 3, 5]

In [ ]:
sys.path.insert(0, os.path.join(base_dir, 'ijepa_modified_k'))
from src.models.vision_transformer import vit_base

In [ ]:
scenarios = [5]
beam_cols = ['unit1_beam_index', 'unit1_beam', 'best_beam']
img_cols = ['unit1_rgb']

In [ ]:
def get_beam_column(df):
    for col in beam_cols:
        if col in df.columns:
            return col
    return None

def load_data():
    records = []
    for s in scenarios:
        csv_path = os.path.join(csv_root, f"scenario{s}", f"scenario{s}.csv")
        img_dir = os.path.join(image_root, f"camera_data_s{s}")

        if not (os.path.exists(csv_path) and os.path.exists(img_dir)):
            continue

        df = pd.read_csv(csv_path)
        beam_col = get_beam_column(df)

        if beam_col is None or "unit1_rgb" not in df.columns:
            continue

        df = df.dropna(subset=[beam_col, "unit1_rgb"]).copy()
        df["img_path"] = df["unit1_rgb"].apply(
            lambda p: os.path.join(img_dir, os.path.basename(str(p))))
        df = df[df["img_path"].apply(os.path.exists)]

        df["beam_label"] = df[beam_col].astype(int)
        df["scenario"] = s

        print(f"Scenario {s}: {len(df)} samples")
        records.append(df[["img_path", "beam_label", "scenario"]])

    if not records:
        print("Total samples: 0")
        return pd.DataFrame(columns=["img_path", "beam_label", "scenario"])

    full = pd.concat(records, ignore_index=True)
    print(f"Total samples: {len(full)}")
    return full

In [ ]:
def split_data(df):
    train_dfs, test_dfs = [], []
    for s in df["scenario"].unique():
        sub = df[df["scenario"] == s]
        n_train = int(len(sub) * 0.7)
        train_dfs.append(sub.iloc[:n_train])
        test_dfs.append(sub.iloc[n_train:])
    return pd.concat(train_dfs, ignore_index=True), pd.concat(test_dfs, ignore_index=True)

#Dataset for loading images and beam labels
class BeamDataset(Dataset):
    def __init__(self, df, beam_to_idx, transform):
        self.df = df.reset_index(drop=True)
        self.beam_to_idx = beam_to_idx
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        try:
            img = Image.open(row["img_path"]).convert("RGB")
        except:
            img = Image.new("RGB", (224, 224))
        label = self.beam_to_idx[row["beam_label"]]
        return self.transform(img), label

In [ ]:
class BeamPredictor(nn.Module):
    def __init__(self, encoder, num_classes, embed_dim=768):
        super().__init__()
        self.encoder = encoder
        self.head = nn.Sequential(
            nn.LayerNorm(embed_dim),
            nn.Linear(embed_dim, 512),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(512, num_classes),)

    def forward(self, x):
        tokens = self.encoder(x)
        if tokens.dim() == 3:
            feats = tokens.mean(dim=1)
        else:
            feats = tokens
        return self.head(feats)

In [ ]:
# Load encoder, freeze all layers, then unfreeze last num_unfreeze blocks
def load_encoder():
    encoder = vit_base(patch_size=16)
    checkpoint = torch.load(checkpoint_path, map_location="cpu")
    state_dict = checkpoint.get("target_encoder", checkpoint.get("encoder", checkpoint))
    state_dict = {k.replace("module.", ""): v for k, v in state_dict.items()}
    encoder.load_state_dict(state_dict, strict=False)

    for param in encoder.parameters():
        param.requires_grad = False

    total_blocks = len(encoder.blocks)
    start_block = total_blocks - num_unfreeze #Freeze blocks 0 to 7, unfreeze 8-11

    for i in range(start_block, total_blocks):
        for param in encoder.blocks[i].parameters():
            param.requires_grad = True

    if hasattr(encoder, "norm"):
        for param in encoder.norm.parameters():
            param.requires_grad = True
    trainable = sum(p.numel() for p in encoder.parameters() if p.requires_grad)
    total = sum(p.numel() for p in encoder.parameters())
    print(f"Trainable backbone params: {trainable:,} / {total:,} ({100.*trainable/total:.1f}%)")
    return encoder

In [ ]:
def build_optimizer(model):
    backbone_params = [p for p in model.encoder.parameters() if p.requires_grad]
    head_params = list(model.head.parameters())
    return torch.optim.AdamW([
        {"params": backbone_params, "lr": lr_backbone},
        {"params": head_params, "lr": lr_head},], weight_decay=weight_decay)


def topk_accuracy(logits, labels):
    results = {}
    for k in top_k:
        k_ = min(k, logits.shape[1])
        topk = logits.topk(k_, dim=1).indices
        correct = topk.eq(labels.unsqueeze(1).expand_as(topk)).any(dim=1).sum().item()
        results[f"top{k}"] = 100.0 * correct / len(labels)
    return results

In [ ]:
def train_model(model, train_loader, test_loader, num_classes, train_df, beam_to_idx):
    model = model.to(device)

    all_labels = train_df["beam_label"].map(beam_to_idx).values
    counts = np.bincount(all_labels, minlength=num_classes)
    weights = torch.tensor( len(all_labels) / (num_classes * np.maximum(counts, 1)),
    dtype=torch.float32, device=device)
    criterion = nn.CrossEntropyLoss(weight=weights)

    optimizer = build_optimizer(model)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    for epoch in range(1, epochs + 1):
        model.train()
        epoch_loss, correct, total = 0.0, 0, 0
        for imgs, labels in train_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            optimizer.zero_grad()
            logits = model(imgs)
            loss = criterion(logits, labels)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            epoch_loss += loss.item()
            correct += (logits.argmax(1) == labels).sum().item()
            total += labels.size(0)

        scheduler.step()

        train_loss = epoch_loss / len(train_loader)
        train_acc = 100 * correct / total

        model.eval()
        with torch.no_grad():
            all_logits = torch.cat([model(imgs.to(device)).cpu() for imgs, _ in test_loader])
            all_labels_eval = torch.cat([labels for _, labels in test_loader])

        accs = topk_accuracy(all_logits, all_labels_eval)

        if epoch % 10 == 0 or epoch == epochs:
            print(f"Epoch {epoch:>3}/{epochs} | Loss: {train_loss:.4f} | "
                f"Train: {train_acc:.2f}% | Top-3: {accs['top3']:.2f}%" )

    return model, all_logits, all_labels_eval

In [ ]:
def plot_results(final_logits, final_labels):
    final_accs = topk_accuracy(final_logits, final_labels)
    ks = [f"Top-{k}" for k in top_k]
    values = [final_accs[f"top{k}"] for k in top_k]
    colors = ["#2196F3", "#4CAF50", "#FF9800", "#F44336"]
    fig, ax = plt.subplots(figsize=(7, 5))
    bars = ax.bar(ks, values, color=colors, width=0.5)
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
                f"{val:.2f}%", ha="center", va="bottom", fontsize=11, fontweight="bold")
    ax.set_ylim(0, 105)
    ax.set_xlabel("Top-k")
    ax.set_ylabel("Accuracy (%)")
    ax.set_title("Beam Prediction - I-JEPA Partial Fine-Tuning (Last 4 Blocks)")
    ax.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(results_dir, "finetune_beam.png"), dpi=150)
    plt.close()

In [ ]:
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(0.2, 0.2, 0.2, 0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),])

test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),])

In [ ]:
df = load_data()
all_beams = sorted(df["beam_label"].unique())
beam_to_idx = {b: i for i, b in enumerate(all_beams)}
num_classes = len(all_beams)
# print(f"Unique beams: {num_classes}")

In [ ]:

train_df, test_df = split_data(df)
# print(f"Train: {len(train_df)} | Test: {len(test_df)}")
train_loader = DataLoader(BeamDataset(train_df, beam_to_idx, train_transform),
                          batch_size=batch_size, shuffle=True, num_workers=16, pin_memory=True)
test_loader = DataLoader(BeamDataset(test_df, beam_to_idx, test_transform),
                         batch_size=batch_size, shuffle=False, num_workers=16, pin_memory=True)

In [ ]:
encoder = load_encoder()
model = BeamPredictor(encoder, num_classes, embed_dim=768)
total_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total trainable parameters: {total_trainable:,}")

In [ ]:
model, final_logits, final_labels = train_model(model, train_loader, test_loader, num_classes, train_df, beam_to_idx)
final_accs = topk_accuracy(final_logits, final_labels)

In [ ]:
print("I-JEPA - Partial Fine-Tuning")
print("="*40)
for k in top_k:
    print(f"Top-{k}: {final_accs[f'top{k}']:.2f}%")
print("="*40)
plot_results(final_logits, final_labels)
print("\nDone. Results saved to:", results_dir)

# Few-shot adaptation on E-flash Dataset
### The I-JEPA encoder is compared to a ResNet-18 baseline

In [ ]:
base_dir = '/content/drive/MyDrive/Colab Notebooks/ENEL 619 07'
episode_path = os.path.join(base_dir, 'E-flash dataset/Cat2/ped_left_to_right/episode_3/npz') #this can be changed to any episode
checkpoint_path = os.path.join(base_dir, 'ijepa_checkpoints/deepsense6G_hpc-latest.pth.tar')
results_dir = os.path.join(base_dir, 'results_eflash')
ijepa_dir = os.path.join(base_dir, 'ijepa_modified_k')
os.makedirs(results_dir, exist_ok=True)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
shot_counts = [10, 20, 30]
num_trials = 5
top_k = [1, 2, 3, 5]
epochs = 100
lr = 1e-4
batch_size = 128
seed = 42
torch.manual_seed(seed)
np.random.seed(seed)
sys.path.insert(0, ijepa_dir)
sys.path.insert(0, os.path.join(ijepa_dir, 'src'))

from models.vision_transformer import vit_base

In [ ]:
#Import the data (Data is stored in .npz format)
image_data = np.load(os.path.join(episode_path, 'image9.npz'), allow_pickle=True)
rf_data = np.load(os.path.join(episode_path, 'rf.npz'), allow_pickle=True)
image_key = list(image_data.keys())[0]
beam_key = list(rf_data.keys())[0]

images = image_data[image_key]
beams = rf_data[beam_key]

In [ ]:
# Convert the 2D beam powers to class labels using argmax function
if beams.ndim == 2:
    beams = beams.argmax(axis=1)
beams = beams.astype(int).flatten()

if images.ndim == 4 and images.shape[-1] != 3:
    images = images.transpose(0, 2, 3, 1)

In [ ]:
unique_beams = sorted(np.unique(beams))
beam_to_idx = {b: i for i, b in enumerate(unique_beams)}
num_classes = len(unique_beams)
labels = np.array([beam_to_idx[b] for b in beams])

In [ ]:
class FrameDataset(Dataset):
    def __init__(self, images, labels, transform):
        self.images = images
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img = Image.fromarray(self.images[idx].astype(np.uint8)).convert('RGB')
        return self.transform(img), int(self.labels[idx])

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),])

In [ ]:
def load_ijepa_encoder():
    encoder = vit_base(patch_size=16)
    checkpoint = torch.load(checkpoint_path, map_location='cpu')
    state_dict = checkpoint.get('target_encoder', checkpoint.get('encoder', checkpoint))
    state_dict = {k.replace('module.', ''): v for k, v in state_dict.items()}
    encoder.load_state_dict(state_dict, strict=False)
    for param in encoder.parameters():
        param.requires_grad = False
    encoder.eval()
    return encoder

def load_resnet18():
    resnet = models.resnet18(pretrained=True)
    encoder = nn.Sequential(*list(resnet.children())[:-1])
    for param in encoder.parameters():
        param.requires_grad = False
    encoder.eval()
    return encoder

ijepa_encoder = load_ijepa_encoder().to(device)
resnet_model = load_resnet18().to(device)

In [ ]:
def extract_features(encoder, images, labels, encoder_type):
    dataset = FrameDataset(images, labels, transform)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=16)

    all_feats, all_labels = [], []
    with torch.no_grad():
        for imgs, lbls in loader:
            if encoder_type == 'ijepa':
                tokens = encoder(imgs.to(device))
                feats = tokens.mean(dim=1) if tokens.dim() == 3 else tokens
            else:
                feats = encoder(imgs.to(device)).squeeze(-1).squeeze(-1)
            all_feats.append(feats.cpu())
            all_labels.append(lbls)

    return torch.cat(all_feats), torch.cat(all_labels)

ijepa_feats, all_labels = extract_features(ijepa_encoder, images, labels, 'ijepa')
resnet_feats, _ = extract_features(resnet_model, images, labels, 'resnet')

In [ ]:
#Support set and query set splitting
def sample_fewshot(feats, labels, n_shots, seed):
    rng = np.random.RandomState(seed)
    support_idx, query_idx = [], []

    for cls in range(num_classes):
        idx = (labels == cls).nonzero(as_tuple=True)[0].tolist()
        if len(idx) == 0:
            continue
        n = min(n_shots, len(idx))
        chosen = rng.choice(idx, n, replace=False).tolist()
        support_idx.extend(chosen)
        query_idx.extend([i for i in idx if i not in chosen])

    return (feats[support_idx], labels[support_idx], feats[query_idx], labels[query_idx])

In [ ]:
class LinearHead(nn.Module):
    def __init__(self, in_dim, num_classes):
        super().__init__()
        self.fc = nn.Linear(in_dim, num_classes)

    def forward(self, x):
        return self.fc(x)

def train_head(support_feats, support_labels, in_dim):
    head = LinearHead(in_dim, num_classes).to(device)
    optimizer = torch.optim.Adam(head.parameters(), lr=lr, weight_decay=1e-4)
    criterion = nn.CrossEntropyLoss()

    dataset = torch.utils.data.TensorDataset(support_feats, support_labels)
    loader = DataLoader(dataset, batch_size=min(32, len(dataset)), shuffle=True)

    head.train()
    for _ in range(epochs):
        for xb, yb in loader:
            optimizer.zero_grad()
            loss = criterion(head(xb.to(device)), yb.to(device))
            loss.backward()
            optimizer.step()
    head.eval()
    return head

In [ ]:
def topk_accuracy(logits, labels):
    results = {}
    for k in top_k:
        k_actual = min(k, logits.shape[1])
        topk_preds = logits.topk(k_actual, dim=1).indices
        correct = topk_preds.eq(labels.unsqueeze(1)).any(1).float().mean().item()
        results[f'top{k}'] = 100.0 * correct
    return results

In [ ]:
ijepa_results = {}
resnet_results = {}

for n_shots in shot_counts:
    sup_ijepa, sup_l, qry_ijepa, qry_l = sample_fewshot(ijepa_feats, all_labels, n_shots, seed)
    sup_resnet, _, qry_resnet, _ = sample_fewshot(resnet_feats, all_labels, n_shots, seed)

    ijepa_head = train_head(sup_ijepa, sup_l, ijepa_feats.shape[1])
    with torch.no_grad():
        ijepa_logits = ijepa_head(qry_ijepa.to(device)).cpu()
    ijepa_accs = topk_accuracy(ijepa_logits, qry_l)
    ijepa_results[n_shots] = ijepa_accs

    resnet_head = train_head(sup_resnet, sup_l, resnet_feats.shape[1])
    with torch.no_grad():
        resnet_logits = resnet_head(qry_resnet.to(device)).cpu()
    resnet_accs = topk_accuracy(resnet_logits, qry_l)
    resnet_results[n_shots] = resnet_accs

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

n_groups = len(shot_counts)
width = 0.18
colors = ['#2196F3', '#4CAF50', '#FF9800', '#F44336']
x = np.arange(n_groups)

for i, k in enumerate(top_k):
    values = [ijepa_results[s][f'top{k}'] for s in shot_counts]
    offset = (i - 2 + 0.5) * width
    bars = axes[0].bar(x + offset, values, width, label=f'Top-{k}', color=colors[i])
    for bar, val in zip(bars, values):
        axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                     f'{val:.1f}', ha='center', va='bottom', fontsize=8)
axes[0].set_xticks(x)
axes[0].set_xticklabels([f'{s}-shot' for s in shot_counts])
axes[0].set_xlabel('Shots per Class')
axes[0].set_ylabel('Accuracy (%)')
axes[0].set_ylim(0, 110)
axes[0].set_title('I-JEPA')
axes[0].legend()
axes[0].grid(axis='y', alpha=0.3)

for i, k in enumerate(top_k):
    values = [resnet_results[s][f'top{k}'] for s in shot_counts]
    offset = (i - 2 + 0.5) * width
    bars = axes[1].bar(x + offset, values, width, label=f'Top-{k}', color=colors[i])
    for bar, val in zip(bars, values):
        axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                     f'{val:.1f}', ha='center', va='bottom', fontsize=8)
axes[1].set_xticks(x)
axes[1].set_xticklabels([f'{s}-shot' for s in shot_counts])
axes[1].set_xlabel('Shots per Class')
axes[1].set_ylabel('Accuracy (%)')
axes[1].set_ylim(0, 110)
axes[1].set_title('ResNet-18')
axes[1].legend()
axes[1].grid(axis='y', alpha=0.3)
plt.suptitle('Few-Shot Beam Prediction - e-FLASH Cat2 Episode 3', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(results_dir, 'fewshot_comparison.png'), dpi=150, bbox_inches='tight')
plt.show()